# Introduction

This notebook tests the use of the Voila package to convert the instrument function calculation notebook into a web application.

To serve a directory of notebooks in Voila run:

`poetry run voila`

Pass:

- `--theme=dark` for a dark theme.
- `--VoilaConfiguration.http_header_envs=Cookie` to pass cookies to the environment (and be able to have access to them in the notebook).
- `--Voila.tornado_settings="{'headers': {'Set-Cookie': 'my_cookie=cookie_value; Path=/; HttpOnly'}}"` to send a cookie `my_cookie` with a value `cookie_value` to the client side.

By default, while running locally, the notebook can be accessed at `http://localhost:8866/voila/render/notebook_name.ipynb?`.
It is also possible to pass query string parameters.
For example, to run this notebook and pass a filename for immediate instrument function computation, access:

`http://localhost:8866/voila/render/voila_instrument_functions.ipynb?filename=name_of_the_file`


In [ ]:
import os
from http import cookies
from urllib.parse import parse_qs
import numpy as np

import plotly.graph_objects as go
from IPython.utils.capture import capture_output
import ipywidgets as widgets
from IPython.display import display

import mascope_lib.runtime as lib_runtime

lib_runtime.init()

from mascope_lib.file_func import get_instrument_type, get_sum_signal
from mascope_lib.peak import gen_peak, fit_n_peaks, segment_spec
from mascope_lib.instrument_functions import (
    process_peak_shapes,
    fit_fwhm,
    fit_resolution_function,
)
from mascope_lib.inst_func_viz import vizualize

import mascope_sdk

In [ ]:
# Reading cookie

cookie = cookies.SimpleCookie()
cookie.load(os.environ.get("HTTP_COOKIE", "Cookie env var not found"))
cookie_dict = {key: morsel.value for key, morsel in cookie.items()}

print(f"The value of the cookie passed: {cookie_dict['my_cookie']}")

In [ ]:
# Getting query parameters

query_string = os.environ.get("QUERY_STRING", "")
parameters = parse_qs(query_string)

for key, val in parameters.items():
    print(f"Param {key} has a value of {val}")
print("query string parameters:", parameters)

In [ ]:
def on_value_change(change):
    if change["new"] == "file":
        filename_input.layout.visibility = "visible"
        mascope_url_input.layout.visibility = "hidden"
        sample_file_id_input.layout.visibility = "hidden"
        sample_file_id_input.value = ""
    elif change["new"] == "api":
        filename_input.layout.visibility = "hidden"
        filename_input.value = ""
        mascope_url_input.layout.visibility = "visible"
        sample_file_id_input.layout.visibility = "visible"

In [ ]:
def get_spectrum_and_instrument():
    try:
        if source_flag.value == "file":
            filename = filename_input.value

            # Extract sum spectrum
            sum_signal = get_sum_signal(filename)
            instrument_type = get_instrument_type(filename)
            mz = sum_signal.mz.values
            spec = sum_signal.values
        else:
            mascope_url = mascope_url_input.value
            sample_file_id = sample_file_id_input.value
            with capture_output():
                sample_file = mascope_sdk.get_sample(mascope_url, sample_file_id)
                file_id = sample_file["sample_file_id"]
                instrument_type = get_instrument_type(sample_file["filename"])
                sample_spectrum = mascope_sdk.get_sample_file_spectrum(
                    mascope_url, file_id
                )
            mz = np.array(sample_spectrum["mz"])
            spec = np.array(sample_spectrum["intensity"])
        return mz, spec, instrument_type
    except Exception as e:
        with result_output:
            result_output.clear_output()
            print("Check inputs")
            print(e)

In [ ]:
def calculate_and_show(change):
    with result_output:
        result_output.clear_output()
        print("Wait and pray...")

    mz, spec, instrument_type = get_spectrum_and_instrument()

    dmz = 0.5
    r_squared_threshold = 0.96
    n_peaks = 100

    # Get x-domain, normalized peak shapes and associated peak positions and FWHMs
    p_x, p_ys, p_mzs, p_fwhms = process_peak_shapes(
        mz, spec, instrument_type, dmz, r_squared_threshold, n_peaks
    )

    # Calculate median peak shape
    p_median = np.median(np.array([p_y for p_y in p_ys]), axis=0)

    peak_shape = {"x": p_x, "y": p_median}

    # Calculate resolution function
    p_mzs = np.array(p_mzs)
    p_fwhms = np.array(p_fwhms)
    p_fwhms_fit = fit_fwhm(instrument_type, p_mzs, p_fwhms)

    # Number of std to filter out outliers in FWHM fit
    ndev = 1

    resolution_function, _ = fit_resolution_function(
        instrument_type, p_mzs, p_fwhms, ndev
    )

    u_list = p_mzs
    # Stack mass ranges
    u_range = np.vstack([u_list - 0.5, u_list + 0.5])
    # Broadcast the u_range array to have the same shape as mz
    u_range = u_range[:, :, np.newaxis]
    # Create boolean masks indicating which elements of spec fall within each range
    mask_u_list = (mz >= u_range[0]) & (mz <= u_range[1])
    mask_u_list = mask_u_list.any(axis=0)
    # Update mz and spec
    new_mz = mz[mask_u_list]
    new_spec = spec[mask_u_list]

    if new_spec.size == 0:
        # Nothing to fit
        specs_to_fit = []
    else:
        if instrument_type == "orbi":
            segmented_indices = segment_spec(new_spec)
            specs_to_fit = [
                (new_mz[chunk], new_spec[chunk]) for chunk in segmented_indices
            ]
        if instrument_type == "tof":
            dmz = 0.5
            specs_to_fit = [
                (
                    new_mz[np.logical_and(new_mz >= u - dmz, new_mz <= u + dmz)],
                    new_spec[np.logical_and(new_mz >= u - dmz, new_mz <= u + dmz)],
                )
                for u in u_list
            ]

    fitted_peaks = []
    for mz_chunk, spec_chunk in specs_to_fit:
        fit, fitted_chunk = fit_n_peaks(
            mz_chunk, spec_chunk, peak_shape, resolution_function, 0.9, max_n_peaks=10
        )
        if fit is None:
            # Nothing to fit
            continue

        for i in fitted_chunk:
            fitted_peaks.append(i)

    fitted_peaks = np.asarray(fitted_peaks)
    fit_poss = fitted_peaks[:, 0]
    fit_heis = fitted_peaks[:, 1]

    subtitles = ("FWHM", "Chosen peak", "Resolution function")

    def update_chosen_peak(trace, points, selector):
        """Update chosen peak in resolution function visualization"""
        if points.xs:
            # Clean peak shapes
            fig_widget.layout.shapes = []
            # Clean annotations
            fig_widget.layout.annotations = [
                i for i in fig_widget.layout.annotations if i.text in subtitles
            ]
            chosen_peak_val = points.xs[0]
            # Calculate chosen peak window width
            dmz = 3 * chosen_peak_val / resolution_function(chosen_peak_val)
            # Mask area around chosen_peak_val
            chosen_peak_mask = (chosen_peak_val - dmz < mz) & (
                mz < chosen_peak_val + dmz
            )
            # Filter spectra window to plot
            mz_window_x = mz[chosen_peak_mask]
            mz_window_y = spec[chosen_peak_mask]
            # Init fitted signal
            fit_y = np.zeros_like(mz_window_x)

            # Fitted peak mask
            fit_peak_mask = (chosen_peak_val - dmz < fit_poss) & (
                fit_poss < chosen_peak_val + dmz
            )
            fit_poss_filt = fit_poss[fit_peak_mask]
            fit_heis_filt = fit_heis[fit_peak_mask]
            # Add fitted peaks to fit_y
            for i, mz_val in enumerate(fit_poss_filt):
                fit_y += gen_peak(
                    mz_window_x,
                    mz_val,
                    fit_heis_filt[i],
                    pres=resolution_function(mz_val),
                    ps=peak_shape,
                )
                fig_widget.add_shape(
                    type="line",
                    x0=mz_val,
                    x1=mz_val,
                    y0=0,
                    y1=fit_heis_filt[i],
                    line=dict(width=2),
                    xref="x2",
                    yref="y2",
                )
                fig_widget.add_annotation(
                    x=mz_val,
                    y=fit_heis_filt[i],
                    text=f"{mz_val:.3f}",
                    showarrow=False,
                    xref="x2",
                    yref="y2",
                    yshift=10,
                )
            # Residuals
            residuals = mz_window_y - fit_y

            fig_widget.update_traces(
                {"x": mz_window_x, "y": mz_window_y},
                selector=lambda trace: trace.name == "Chosen peak",
            )
            fig_widget.update_traces(
                {"x": mz_window_x, "y": fit_y},
                selector=lambda trace: trace.name == "Fitted signal",
            )
            fig_widget.update_traces(
                {"x": mz_window_x, "y": residuals},
                selector=lambda trace: trace.name == "Residuals",
            )

    fig = vizualize(p_mzs, p_fwhms, p_fwhms_fit, ndev, resolution_function)

    fig_widget = go.FigureWidget(fig)

    fig_widget.data[4].on_click(update_chosen_peak)
    fig_widget.data[5].on_click(update_chosen_peak)

    with result_output:
        result_output.clear_output()
        display(fig_widget)

In [ ]:
# Create the widgets
source_flag = widgets.Dropdown(
    options=["file", "api"], value="file", description="Source:"
)

filename_input = widgets.Text(
    placeholder="Enter filename", layout=widgets.Layout(visibility="visible")
)

mascope_url_input = widgets.Text(
    placeholder="Enter Mascope URL", layout=widgets.Layout(visibility="hidden")
)

sample_file_id_input = widgets.Text(
    placeholder="Enter Sample File ID", layout=widgets.Layout(visibility="hidden")
)

calculate_button = widgets.Button(description="Calculate")
calculate_button.on_click(calculate_and_show)

result_output = widgets.Output()

display(
    *[
        source_flag,
        filename_input,
        mascope_url_input,
        sample_file_id_input,
        calculate_button,
        result_output,
    ]
)

# Connect the dropdown to the input field visibility
source_flag.observe(on_value_change, names="value")

try:
    # Handle positive polarity
    if parameters["filename"][0].endswith("_ "):
        parameters["filename"][0] = parameters["filename"][0].replace("_ ", "_+")
    print(parameters["filename"][0][-2:])
    # Check if filename passed
    filename_input.value = parameters["filename"][0]
    # "Press" calculate button
    calculate_and_show(calculate_button)
except KeyError:
    pass